# Exploring SesamEO API for Download and Visualization

## Rotterdam Use Case

### Set your SesamEO API key
Follow instructions [here](https://platform.destine.eu/docs/sesameo/doc/index.html#api-key) to obtain an API key.

Additionally, you need to configure your API key with the providers (for example CDSE, Microsoft, etc.)

You will be prompted to enter your API key with hidden input in the next code cell.

In [ ]:
from getpass import getpass
import zipfile
from pathlib import Path
from urllib.parse import quote

import matplotlib.pyplot as plt
import requests
import xarray as xr

In [ ]:
# Global above ground biomass
download_url = "https://api.sesameo.destine.eu/odata/v1/Collections('hgb')/Products('MICROSOFT:hgb')/Links('Global%20above-ground%20biomass')/$value"

In [ ]:
# Example from your curl command (replace with your own links as needed).
API_KEY = getpass("SesamEO API key: ").strip()
if not API_KEY:
    raise ValueError("API key cannot be empty.")

# download_url = (
#     f"{API_BASE}/Collections('{collection_q}')/Products('{product_q}')/Links('DownloadLink')/$value"
# )

output_zip = Path("downloads") / "sesameo_product.zip"
extract_dir = Path("downloads") / "sesameo_product"
output_zip.parent.mkdir(parents=True, exist_ok=True)
extract_dir.mkdir(parents=True, exist_ok=True)

download_url

In [ ]:
def download_file(url: str, api_key: str, output_path: Path, timeout: int = 180) -> Path:
    headers = {"X-API-KEY": api_key}
    with requests.get(url, headers=headers, stream=True, timeout=timeout) as response:
        response.raise_for_status()
        with output_path.open("wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    return output_path

In [ ]:
downloaded_file = download_file(download_url, API_KEY, output_zip)
print(f"Downloaded: {downloaded_file}")

with zipfile.ZipFile(downloaded_file, "r") as zf:
    zf.extractall(extract_dir)

print(f"Extracted to: {extract_dir}")

### Preview files and open a dataset

In [ ]:
all_files = sorted([p for p in extract_dir.rglob("*") if p.is_file()])
for p in all_files[:20]:
    print(p)

nc_files = [p for p in all_files if p.suffix.lower() in {".nc", ".nc4"}]
if not nc_files:
    raise FileNotFoundError("No NetCDF file found in extracted product.")

dataset_path = nc_files[0]
print(f"Using dataset: {dataset_path}")
ds = xr.open_dataset(dataset_path)
ds

In [ ]:
# Pick the first variable with at least 2 dimensions and plot a simple slice.
candidate_vars = [name for name, da in ds.data_vars.items() if da.ndim >= 2]
if not candidate_vars:
    raise ValueError("No plottable variable (ndim >= 2) found in dataset.")

var_name = candidate_vars[0]
da = ds[var_name]

while da.ndim > 2:
    da = da.isel({da.dims[0]: 0})

plt.figure(figsize=(10, 5))
da.plot(cmap="viridis")
plt.title(f"{var_name} (first 2D slice)")
plt.tight_layout()
plt.show()